In [47]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

In [ ]:
URL = "http://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read("bank-additional.zip")))
df = pd.read_csv(inner.open("bank-additional/bank-additional.csv"), sep=";")
print(df.shape)

(4119, 21)


In [49]:
df["y"].value_counts()

y
no     3668
yes     451
Name: count, dtype: int64

In [50]:
print(round((df["y"] == "yes").mean(), 4))

0.1095


In [51]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,...,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,...,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,...,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,...,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   str    
 2   marital         4119 non-null   str    
 3   education       4119 non-null   str    
 4   default         4119 non-null   str    
 5   housing         4119 non-null   str    
 6   loan            4119 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   float64
 18 

In [53]:
df.isnull().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [54]:
df.dtypes

age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

In [55]:
df.groupby("y")["duration"].mean().round(1)

y
no     219.4
yes    560.8
Name: duration, dtype: float64

In [56]:
df = df.drop(columns=["duration"])

In [57]:
df.shape

(4119, 20)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X = df.drop(columns=["y"])
y = df["y"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

print(clf.score(X_test, y_test))
print(clf.score(X_train, y_train))

0.8349514563106796
1.0


In [73]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
)

bag.fit(X_train, y_train)

y_proba = bag.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_proba)

print(f"ROC AUC: {auc:.4f}")

ROC AUC: 0.7560


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    max_features="sqrt",
    oob_score=True,
    criterion="gini",
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

print(rf.score(X_test, y_test))
print(rf.oob_score_)

0.8992718446601942
0.8986342943854325


In [79]:
from sklearn.metrics import confusion_matrix

y_pred = rf.predict(X_test)

matrix = confusion_matrix(y_test, y_pred)

print(matrix)

[[730   4]
 [ 79  11]]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

y_proba = rf.predict_proba(X_test)[:, 1]

print(f"ROC AUC -> {roc_auc_score(y_test, y_proba):.4f}")

              precision    recall  f1-score   support

          no       0.90      0.99      0.95       734
         yes       0.73      0.12      0.21        90

    accuracy                           0.90       824
   macro avg       0.82      0.56      0.58       824
weighted avg       0.88      0.90      0.87       824

ROC AUC -> 0.7798


In [99]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(
    n_estimators=20,
    random_state=42,
)

ada.fit(X_train, y_train)

print(ada.score(X_test, y_test))
print(ada.score(X_train, y_train))

y_proba = ada.predict_proba(X_test)[:, 1]

print(f"ROC AUC -> {roc_auc_score(y_test, y_proba):.4f}")

0.9029126213592233
0.9040971168437025
ROC AUC -> 0.7837


In [102]:
for x in [10, 20, 50, 100, 140, 200, 500]:
    ada = AdaBoostClassifier(
        n_estimators=x,
        random_state=42,
    )
    ada.fit(X_train, y_train)
    y_proba = ada.predict_proba(X_test)[:, 1]
    print(f"ROC AUC -> {roc_auc_score(y_test, y_proba):.4f}")

ROC AUC -> 0.7673
ROC AUC -> 0.7837
ROC AUC -> 0.7804
ROC AUC -> 0.7753
ROC AUC -> 0.7875
ROC AUC -> 0.7864
ROC AUC -> 0.7822
